### Manipulating models to make them able to grow in community

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')

    model1 = reframed.load_cbmodel("../models/species_used/Bacteroides_ovatus_ATCC_8483.xml")
    model2 = reframed.load_cbmodel("../models/species_used/Bacteroides_thetaiotaomicron_VPI_5482.xml")

In [ ]:
import copy

**Adding transport and metabolites for sink reactions**

In [ ]:
for model_i in [model1,model2]:
    sink_reactions = [rxn for rxn in model_i.get_exchange_reactions() if "sink" in rxn]
    extra_reactions = ["R_pbiosynthesis","R_rtranscription","R_dreplication"]
    
    sink_reactions.extend(extra_reactions)
    for rxn in sink_reactions:
        if rxn in extra_reactions:
            met_c_id = model_i.reactions[rxn].get_products()[0]
        else:
            met_c_id = model_i.reactions[rxn].get_substrates()[0]

        met_e_id = copy.copy(met_c_id)
        met_e_id = met_e_id.replace("_c_","_e_")
        met_e = reframed.Metabolite(met_e_id,model_i.metabolites[met_c_id].name,compartment=model_i.compartments.e.id)

        model_i.add_metabolite(met_e)
        model_i.add_reaction_from_str(f"R_{met_c_id[2:-13]}_sink: {met_c_id} <-> {met_e_id}")

        print(model_i.reactions[f"R_{met_c_id[2:-13]}_sink"])
    

R_s_sink: M_s__91__c__93__ <-> M_s__91__e__93__
R_thissh_sink: M_thissh__91__c__93__ <-> M_thissh__91__e__93__
R_proteinsynth_sink: M_proteinsynth__91__c__93__ <-> M_proteinsynth__91__e__93__
R_rnatrans_sink: M_rnatrans__91__c__93__ <-> M_rnatrans__91__e__93__
R_dnarep_sink: M_dnarep__91__c__93__ <-> M_dnarep__91__e__93__
R_gthrd_sink: M_gthrd__91__c__93__ <-> M_gthrd__91__e__93__
R_s_sink: M_s__91__c__93__ <-> M_s__91__e__93__
R_thissh_sink: M_thissh__91__c__93__ <-> M_thissh__91__e__93__
R_proteinsynth_sink: M_proteinsynth__91__c__93__ <-> M_proteinsynth__91__e__93__
R_rnatrans_sink: M_rnatrans__91__c__93__ <-> M_rnatrans__91__e__93__
R_dnarep_sink: M_dnarep__91__c__93__ <-> M_dnarep__91__e__93__


In [ ]:
model1.reactions.R_pbiosynthesis

R_pbiosynthesis:  --> M_proteinsynth__91__c__93__

**Adding transport and metabolites for demand reactions**

In [ ]:
for model_i in [model1,model2]:
    for rxn in [rxn for rxn in model_i.get_exchange_reactions() if "R_DM" in rxn]:
        if "R_DM_atp_c" in rxn:
            continue
            
        print(model_i.reactions[rxn])
        met_c_id = model_i.reactions[rxn].get_substrates()[0]

        met_e_id = copy.copy(met_c_id)
        met_e_id = met_e_id.replace("_c_","_e_")
        met_e = reframed.Metabolite(met_e_id,model_i.metabolites[met_c_id].name,compartment=model_i.compartments.e.id)
        model_i.add_metabolite(met_e)
        
        model_i.add_reaction_from_str(f"R_{met_c_id[2:-13]}_DM: {met_c_id} --> {met_e_id}")
        print(model_i.reactions[f"R_{met_c_id[2:-13]}_DM"])
    
    model_i.update()

R_DM_2HYMEPH: M_2hymeph__91__c__93__ --> 
R_2hymeph_DM: M_2hymeph__91__c__93__ --> M_2hymeph__91__e__93__
R_DM_4HBA: M_4hba__91__c__93__ --> 
R_4hba_DM: M_4hba__91__c__93__ --> M_4hba__91__e__93__
R_DM_AMOB: M_amob__91__c__93__ --> 
R_amob_DM: M_amob__91__c__93__ --> M_amob__91__e__93__
R_DM_adocbl__40__c__41__: M_adocbl__91__c__93__ --> 
R_adocbl_DM: M_adocbl__91__c__93__ --> M_adocbl__91__e__93__
R_DM_btn: M_btn__91__c__93__ --> 
R_btn_DM: M_btn__91__c__93__ --> M_btn__91__e__93__
R_DM_dad_5: M_dad_5__91__c__93__ --> 
R_dad_5_DM: M_dad_5__91__c__93__ --> M_dad_5__91__e__93__
R_DM_kdo2lipid4L__40__c__41__: M_kdo2lipid4L__91__c__93__ --> 
R_kdo2lipid4L_DM: M_kdo2lipid4L__91__c__93__ --> M_kdo2lipid4L__91__e__93__
R_DM_pheme__40__c__41__: M_pheme__91__c__93__ --> 
R_pheme_DM: M_pheme__91__c__93__ --> M_pheme__91__e__93__
R_DM_this__91__c__93__: M_this__91__c__93__ --> 
R_this_DM: M_this__91__c__93__ --> M_this__91__e__93__
R_DM_4HBA: M_4hba__91__c__93__ --> 
R_4hba_DM: M_4hba__91__c__93

In [ ]:
model1.reactions.R_biomass536.stoichiometry.pop("M_biomass__91__c__93__")

1.0

In [ ]:
model2.reactions.R_biomass345.stoichiometry.pop("M_biomass__91__c__93__")

1.0

#### Remove demand reactions and add new exchange reactions

In [ ]:
for model_i in [model1,model2]:
    dm_replace = [rxn for rxn in model_i.get_exchange_reactions() if "R_DM" in rxn]
    for rxn in dm_replace:
        if "R_DM_atp_c" in rxn:
            continue
        
        print(model_i.reactions[rxn].get_substrates()[0])
        met_c_id = model_i.reactions[rxn].get_substrates()[0]
        met_e_id = copy.copy(met_c_id)
        met_e_id = met_e_id.replace("_c_","_e_")
        
        reaction = reframed.CBReaction("R_EX_"+met_e_id[2:],stoichiometry = {met_e_id:-1},reversible=False,reaction_type=reframed.ReactionType.EXCHANGE)
        model_i.add_reaction(reaction)
        model_i.remove_reaction(rxn)
        #model_i.add_reaction_from_str(f"R_{met_c_id[2:-13]}_DM: {met_c_id} --> {met_e_id}")

M_2hymeph__91__c__93__
M_4hba__91__c__93__
M_amob__91__c__93__
M_adocbl__91__c__93__
M_btn__91__c__93__
M_dad_5__91__c__93__
M_kdo2lipid4L__91__c__93__
M_pheme__91__c__93__
M_this__91__c__93__
M_4hba__91__c__93__
M_5mtr__91__c__93__
M_amob__91__c__93__
M_btn__91__c__93__
M_clpn180__91__c__93__
M_clpnai17__91__c__93__
M_clpni17__91__c__93__
M_dad_5__91__c__93__
M_kdo2lipid4L__91__c__93__
M_teich_45_BS__91__c__93__
M_this__91__c__93__


#### Remove sink reactions and add new exchange reactions

In [ ]:
for model_i in [model1,model2]:
    
    sink_reactions = [rxn for rxn in model_i.get_exchange_reactions() if "sink" in rxn]
    extra_reactions = ["R_pbiosynthesis","R_rtranscription","R_dreplication"]
    
    sink_reactions.extend(extra_reactions)
    for rxn in sink_reactions:
        if rxn in extra_reactions:
            met_c_id = model_i.reactions[rxn].get_products()[0]
        else:
            met_c_id = model_i.reactions[rxn].get_substrates()[0]
       
        met_e_id = copy.copy(met_c_id)
        met_e_id = met_e_id.replace("_c_","_e_")
        
        reaction = reframed.CBReaction("R_EX_"+met_e_id[2:],stoichiometry = {met_e_id:-1},reversible=True,reaction_type=reframed.ReactionType.EXCHANGE)
        model_i.add_reaction(reaction)
        print(reaction)
        model_i.remove_reaction(rxn)
        #model_i.add_reaction_from_str(f"R_{met_c_id[2:-13]}_DM: {met_c_id} --> {met_e_id}")

R_EX_s__91__e__93__: M_s__91__e__93__ <-> 
R_EX_thissh__91__e__93__: M_thissh__91__e__93__ <-> 
R_EX_proteinsynth__91__e__93__: M_proteinsynth__91__e__93__ <-> 
R_EX_rnatrans__91__e__93__: M_rnatrans__91__e__93__ <-> 
R_EX_dnarep__91__e__93__: M_dnarep__91__e__93__ <-> 
R_EX_gthrd__91__e__93__: M_gthrd__91__e__93__ <-> 
R_EX_s__91__e__93__: M_s__91__e__93__ <-> 
R_EX_thissh__91__e__93__: M_thissh__91__e__93__ <-> 
R_EX_proteinsynth__91__e__93__: M_proteinsynth__91__e__93__ <-> 
R_EX_rnatrans__91__e__93__: M_rnatrans__91__e__93__ <-> 
R_EX_dnarep__91__e__93__: M_dnarep__91__e__93__ <-> 


In [ ]:
model1.update()
model2.update()

In [ ]:
reframed.save_cbmodel(model1,f"../models/manipulated_models/{model1.id}.xml")
reframed.save_cbmodel(model2,f"../models/manipulated_models/{model2.id}.xml")

#### Test the growth of the individual models

In [ ]:
env = reframed.Environment.from_reactions(lb_ex_rxns_old)

env.apply(model1, inplace=True, exclusive=True, warning=False)

sol = reframed.pFBA(model1)


In [ ]:
lb_ex_rxns_old[0]

'R_EX_adn__40__e__41__'

In [ ]:
sol.values["R_biomass536"]

0.0

In [ ]:
env.apply(model2, inplace=True, exclusive=True, warning=False)

sol = reframed.pFBA(model2)


In [ ]:
model2.biomass_reaction

'R_biomass345'

In [ ]:
sol.values["R_biomass345"]

0.0

In [ ]:
test_lb = list(copy.copy(lb_ex_rxns_old))

In [ ]:
test_lb.append("R_EX_thissh__91__e__93__")
test_lb.append("R_EX_gthrd__91__e__93__")
test_lb.append("R_EX_s__91__e__93__")
test_lb.append("R_EX_proteinsynth__91__e__93__")
test_lb.append("R_EX_rnatrans__91__e__93__")
test_lb.append("R_EX_dnarep__91__e__93__")


In [ ]:
env2= reframed.Environment.from_reactions(test_lb)
env2.apply(model1, inplace=True, exclusive=True, warning=False)
sol = reframed.pFBA(model1)

In [ ]:
sol.values["R_biomass536"]

2.0976637499932598

In [ ]:
env2.apply(model2, inplace=True, exclusive=True, warning=False)

sol = reframed.pFBA(model2)


In [ ]:
sol.values["R_biomass345"]

2.134772368679991

#### Test growth in community

In [ ]:
lb_ex_rxns.append("R_EX_thissh__91__e__93__")
lb_ex_rxns.append("R_EX_gthrd__91__e__93__")
lb_ex_rxns.append("R_EX_s__91__e__93__")
lb_ex_rxns.append("R_EX_proteinsynth__91__e__93__")
lb_ex_rxns.append("R_EX_rnatrans__91__e__93__")
lb_ex_rxns.append("R_EX_dnarep__91__e__93__")

In [ ]:
comm = reframed.Community("test",[model1,model2],copy_models=True)

env = reframed.Environment.from_reactions(lb_ex_rxns)
env.apply(comm.merged_model, inplace=True, exclusive=True, warning=False)

sol = reframed.pFBA(comm.merged_model)

In [ ]:
sol.values["community_growth"]

2.159784218927665

#### Finding blocked reactions

**In not manipulated model**

In [ ]:
with warnings.catch_warnings():
    warnings.filterwarnings('ignore')

    model1_n = reframed.load_cbmodel("../models/species_used/Bacteroides_ovatus_ATCC_8483.xml")
    model2_n = reframed.load_cbmodel("../models/species_used/Bacteroides_thetaiotaomicron_VPI_5482.xml")

In [ ]:
comm2 = reframed.Community("test",[model1_n,model2_n],copy_models=True)

env_complete.apply(comm2.merged_model, inplace=True, exclusive=True, warning=False)

In [ ]:
met_one_reaction = []
for met in comm2.merged_model.metabolites:
    rxns = comm2.merged_model.get_metabolite_reactions(met)
    if len(rxns)==1:
        met_one_reaction.append(rxns[0])

In [ ]:
sol = reframed.FVA(comm2.merged_model,obj_frac=0)

/Users/idunmariaburgos/opt/anaconda3/envs/rcell_rec/lib/python3.8/site-packages/reframed/cobra/variability.py:56: UserWarning: Infeasible solution status
  warn('Infeasible solution status')
/Users/idunmariaburgos/opt/anaconda3/envs/rcell_rec/lib/python3.8/site-packages/reframed/cobra/variability.py:58: UserWarning: Unknown solution status
  warn('Unknown solution status')
/Users/idunmariaburgos/opt/anaconda3/envs/rcell_rec/lib/python3.8/site-packages/reframed/cobra/variability.py:76: UserWarning: Unknown solution status
  warn('Unknown solution status')
/Users/idunmariaburgos/opt/anaconda3/envs/rcell_rec/lib/python3.8/site-packages/reframed/cobra/variability.py:74: UserWarning: Infeasible solution status
  warn('Infeasible solution status')


In [ ]:
none_reactions = [rxn for rxn,value in sol.items() if (value[0] is None) or (value[1] is None)]

In [ ]:
new_sol = {rxn:value for rxn,value in sol.items() if rxn not in none_reactions}

blocked_reactions = [rxn for rxn,value in new_sol.items() if ((abs(value[0])<1e-6) and (abs(value[1])<1e-6))]

In [ ]:
len(blocked_reactions)+len(none_reactions)

1330

In [ ]:
len(met_one_reaction)

503

In [ ]:
len(set(met_one_reaction).intersection(set(blocked_reactions)))

399

In [ ]:
len(set(met_one_reaction).intersection(set(none_reactions)))

14

In [ ]:
len(none_reactions)

131

In [ ]:
len(comm2.merged_model.reactions)

4802

**In manipulated model**

In [ ]:
met_one_reaction = []
for met in comm.merged_model.metabolites:
    rxns = comm.merged_model.get_metabolite_reactions(met)
    if len(rxns)==1:
        met_one_reaction.append(rxns[0])

In [ ]:
sol = reframed.FVA(comm.merged_model,obj_frac=0)

In [ ]:
none_reactions = [rxn for rxn,value in sol.items() if (value[0] is None) or (value[1] is None)]

In [ ]:
new_sol = {rxn:value for rxn,value in sol.items() if rxn not in none_reactions}

blocked_reactions = [rxn for rxn,value in new_sol.items() if ((abs(value[0])<1e-6) and (abs(value[1])<1e-6))]

In [ ]:
len(blocked_reactions)+len(none_reactions)

1137

In [ ]:
len(met_one_reaction)

489

In [ ]:
len(set(met_one_reaction).intersection(set(blocked_reactions)))

397

In [ ]:
len(set(met_one_reaction).intersection(set(none_reactions)))

7

In [ ]:
len(comm.merged_model.reactions)

4842

#### Find rxns also blocked in individual models

In [ ]:
env_complete_m1 = reframed.Environment.complete(model1).apply(model1,inplace=True,exclusive=True)
env_complete_m2 = reframed.Environment.complete(model2).apply(model2,inplace=True,exclusive=True)

In [ ]:
sol_m1 = reframed.FVA(model1,obj_frac=0)
sol_m2 = reframed.FVA(model2,obj_frac=0)

In [ ]:
model1.id

'M_Bacteroides_ovatus_ATCC_8483'

In [ ]:
model2.id

'M_Bacteroides_thetaiotaomicron_VPI_5482'

In [ ]:
sol_m1_new = {rxn+"_"+model1.id:values for rxn,values in sol_m1.items()}
sol_m2_new = {rxn+"_"+model2.id:values for rxn,values in sol_m2.items()}
sol_m1_m2 = {**sol_m1_new,**sol_m2_new}

In [ ]:
sol_m1_m2_s = pd.Series(sol_m1_m2)
sol_m1_m2_s.name = "m1+m2"

In [ ]:
sol_merged_s = pd.Series(sol)
sol_merged_s.name = "merged"

In [ ]:
combined_df = pd.concat([sol_m1_m2_s,sol_merged_s],axis=1)

combined_df.dropna(inplace=True)
combined_df

,m1+m2,merged
R_12PPD_Stex_M_Bacteroides_ovatus_ATCC_8483,"[-2770.0, 0.0]","[-2770.0, 0.0]"
R_12PPDtpp_M_Bacteroides_ovatus_ATCC_8483,"[-2770.0, 0.0]","[-2770.0, 0.0]"
R_1HIBUPGLU_St2pp_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[0.0, 10.0]"
R_1HIBUPGLU_Stex_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[0.0, 10.0]"
R_1HIBUP_S_GLCAASE_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[0.0, 10.0]"
...,...,...
R_clpni17_DM_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 9.99999999998844]","[0.0, 9.999999999999954]"
R_dad_5_DM_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 10.0]","[0.0, 10.0]"
R_kdo2lipid4L_DM_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 10.000000000132715]","[0.0, 10.000000000000268]"
R_teich_45_BS_DM_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.4444444444534992]","[0.0, 0.4444444444444452]"


In [ ]:
blocked_or_none = copy.copy(blocked_reactions)
blocked_or_none.extend(none_reactions)

In [ ]:
different_sol_df =  combined_df[combined_df["m1+m2"]!=combined_df["merged"]]

different_sol_df = different_sol_df[different_sol_df.index.isin(blocked_or_none)]
different_sol_df.head(30)

,m1+m2,merged
R_2OBUTtex_M_Bacteroides_ovatus_ATCC_8483,"[-1681171.4705882382, 10.0]","[None, 10.0]"
R_2OH_CBZ_GLC_GLCAASE_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[None, 10.0]"
R_2OH_CBZ_GLC_GLCAASEepp_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[None, 10.0]"
R_2OH_CBZ_GLCt2pp_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[None, 10.0]"
R_2OH_CBZ_GLCtex_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[None, 10.0]"
R_2OH_CBZtepp_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[None, 10.0]"
R_2OH_CBZtex_M_Bacteroides_ovatus_ATCC_8483,"[-10.0, 0.0]","[None, 0.0]"
R_2OH_MTZ_GLC_GLCAASE_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[None, 10.0]"
R_2OH_MTZ_GLC_GLCAASEepp_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[None, 10.0]"
R_2OH_MTZ_GLCt2pp_M_Bacteroides_ovatus_ATCC_8483,"[0.0, 10.0]","[None, 10.0]"


In [ ]:
different_sol_df[different_sol_df.index.str.contains("_DM_M|sink")]

,m1+m2,merged
R_gthrd_sink_M_Bacteroides_thetaiotaomicron_VPI_5482,"[-0.06788787374385921, 0.0]","[0.0, 0.0]"
R_thissh_sink_M_Bacteroides_thetaiotaomicron_VPI_5482,"[-0.06788787374370793, 0.0]","[0.0, 0.0]"
R_4hba_DM_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374381214]","[0.0, 0.0]"
R_this_DM_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.0678878737432866]","[0.0, 0.0]"


In [ ]:
for rxn in different_sol_df[different_sol_df.index.str.contains("_DM_M|sink")].index:
    print(comm.merged_model.reactions[rxn])
    external = comm.merged_model.reactions[rxn].get_products()[0]
    cytosolic = comm.merged_model.reactions[rxn].get_substrates()[0]
    
    print(f"\t external reactions:{comm.merged_model.get_metabolite_reactions(external)}\n")
    print(f"\t cytosolic reactions:{comm.merged_model.get_metabolite_reactions(cytosolic)}\n")
    
    

R_gthrd_sink_M_Bacteroides_thetaiotaomicron_VPI_5482: M_gthrd__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482 <-> M_gthrd__91__e__93__
	 external reactions:['R_gthrd_sink_M_Bacteroides_thetaiotaomicron_VPI_5482', 'R_EX_gthrd__91__e__93__']

	 cytosolic reactions:['R_LGTHL_M_Bacteroides_thetaiotaomicron_VPI_5482', 'R_biomass345_M_Bacteroides_thetaiotaomicron_VPI_5482', 'R_gthrd_sink_M_Bacteroides_thetaiotaomicron_VPI_5482']

R_thissh_sink_M_Bacteroides_thetaiotaomicron_VPI_5482: M_thissh__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482 <-> M_thissh__91__e__93__
	 external reactions:['R_thissh_sink_M_Bacteroides_ovatus_ATCC_8483', 'R_thissh_sink_M_Bacteroides_thetaiotaomicron_VPI_5482', 'R_EX_thissh__91__e__93__']

	 cytosolic reactions:['R_DXYTST_M_Bacteroides_thetaiotaomicron_VPI_5482', 'R_thissh_sink_M_Bacteroides_thetaiotaomicron_VPI_5482']

R_4hba_DM_M_Bacteroides_thetaiotaomicron_VPI_5482: M_4hba__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482 --> M_4hba__91__e__93__


**Find essential reactions**

In [ ]:
essential_m2 = []

for rxn in model2.reactions:
    knockout = reframed.reaction_knockout(model2,[rxn])
    
    if knockout.fobj<1e-6:
        essential_m2.append(rxn)

In [ ]:
different_sol_df[ (different_sol_df.index.isin([rxn+"_"+model2.id for rxn in essential_m2]))]

,m1+m2,merged
R_ACBIPGT_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374373186]","[0.0, 0.0]"
R_ACPS1_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374373276]","[0.0, 0.0]"
R_ADCL_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.20366362123014925]","[0.0, 0.0]"
R_ADCS_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.20366362123014925]","[0.0, 0.0]"
R_ADK2_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374338304]","[0.0, 0.0]"
...,...,...
R_ZN2tex_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787373807281]","[0.0, 0.0]"
R_r0339_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.10183181061537483]","[0.0, 0.0]"
R_r2137pp_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374374732]","[0.0, 0.0]"
R_biomass345_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 21.92406708987959]","[0.0, 0.0]"


In [ ]:
transport_m2 = [rxn+"_"+model2.id for rxn in model2.get_reactions_by_type(reframed.ReactionType.TRANSPORT)]

different_sol_df[ (different_sol_df.index.isin([rxn+"_"+model2.id for rxn in essential_m2])) & (different_sol_df.index.isin(transport_m2))]

,m1+m2,merged
R_CA2tex_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374383126]","[0.0, 0.0]"
R_CAt4ipp_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374393003]","[0.0, 0.0]"
R_CLtex_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374194899]","[0.0, 0.0]"
R_COBALT2tex_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374367752]","[0.0, 0.0]"
R_COBALTt5pp_M_Bacteroides_thetaiotaomicron_VPI_5482,"[-0.06788787374387625, 0.0]","[0.0, 0.0]"
R_CU2tex_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374358193]","[0.0, 0.0]"
R_FE2abcpp_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.20349093914124194]","[0.0, 0.0]"
R_FE2tex_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.20349093914124194]","[0.0, 0.0]"
R_HEMEtipp_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787374363149]","[0.0, 0.0]"
R_Ktex_M_Bacteroides_thetaiotaomicron_VPI_5482,"[0.0, 0.06788787373807281]","[0.0, 0.0]"


In [ ]:
set(met_one_reaction).intersection(set([rxn+"_"+model2.id for rxn in essential_m2]))

{'R_biomass345_M_Bacteroides_thetaiotaomicron_VPI_5482'}

In [ ]:
[rxn for rxn in comm.merged_model.reactions if "biomass" in rxn]

['R_biomass536_M_Bacteroides_ovatus_ATCC_8483',
 'R_biomass345_M_Bacteroides_thetaiotaomicron_VPI_5482']

In [ ]:
comm.merged_model.biomass_reaction

'community_growth'

In [ ]:
interesting_reactions = different_sol_df[ (different_sol_df.index.isin([rxn+"_"+model2.id for rxn in essential_m2]))].index

blocked_reactions_manip = {}
checked_met = []
for rxn in interesting_reactions:
    mets = comm.merged_model.reactions[rxn].get_substrates()

    for met in mets:
        if met in checked_met:
            continue
            
        print(met)
        checked_met.append(met)
        temp_model = comm.merged_model.copy()
        temp_model.add_reaction_from_str(f"R_{met[2:]}_extra_sink: met <->")

M_adocbip__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_gtp__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_h__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_apoACP__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_coa__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_4adcho__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_chor__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_gln_L__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_amp__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_pppi__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_adocbi__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_agdpcbi__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_rdmbzi__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_adphep_DD__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_ala_D__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_atp__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482
M_ala_L__91__c__93___M_Bac

In [ ]:
temp_model.get_metabolite_reactions("M_proteinsynth__91__c__93___M_Bacteroides_thetaiotaomicron_VPI_5482")

['R_biomass345_M_Bacteroides_thetaiotaomicron_VPI_5482']

In [ ]:
model2.get_metabolite_reactions("M_proteinsynth__91__c__93__")

['R_pbiosynthesis', 'R_biomass345']

In [ ]:
model2.reactions.R_pbiosynthesis

R_pbiosynthesis:  <-> M_proteinsynth__91__c__93__ [-10.0, inf]

In [ ]:
[rxn for rxn in temp_model.reactions if "R_pbiosyn" in rxn]

[]

In [ ]:
"R_pbiosynthesis" in model2.get_exchange_reactions()

True

**Example of a CarveMe model**

In [ ]:
model_test = reframed.load_cbmodel("../../Ruminiclostridium_cellulolyticum_model/Ruminiclostridium-cellullolyticum-model-final/models/RcH10_final.xml")

In [ ]:
model_test.reactions.R_Growth

R_Growth: 0.000223 M_10fthf_c + 0.513689 M_ala__L_c + 0.000223 M_amet_c + 0.295792 M_arg__L_c + 0.241055 M_asn__L_c + 0.241055 M_asp__L_c + 29.54 M_atp_c + 0.005205 M_ca2_c + 0.005205 M_cl_c + 0.000576 M_coa_c + 0.0001 M_cobalt2_c + 0.133508 M_ctp_c + 0.000709 M_cu2_c + 0.09158 M_cys__L_c + 0.026166 M_datp_c + 0.027017 M_dctp_c + 0.027017 M_dgtp_c + 0.026166 M_dttp_c + 0.000223 M_fad_c + 0.006715 M_fe2_c + 0.007808 M_fe3_c + 0.26316 M_gln__L_c + 0.26316 M_glu__L_c + 0.612638 M_gly_c + 0.001 M_gtca1_45_BS_c + 0.001 M_gtca2_45_BS_c + 0.001 M_gtca3_45_BS_c + 0.215096 M_gtp_c + 48.601527 M_h2o_c + 0.094738 M_his__L_c + 0.290529 M_ile__L_c + 0.195193 M_k_c + 0.450531 M_leu__L_c + 5e-05 M_lipo1_24_BS_c + 5e-05 M_lipo2_24_BS_c + 5e-05 M_lipo3_24_BS_c + 5e-05 M_lipo4_24_BS_c + 0.343161 M_lys__L_c + 0.153686 M_met__L_c + 0.008675 M_mg2_c + 0.000223 M_mlthf_c + 0.000691 M_mn2_c + 0.0001 M_mql8_c + 0.001831 M_nad_c + 0.000447 M_nadp_c + 0.01 M_peptido_BS_c + 0.185265 M_phe__L_c + 0.221055 M_pro__

In [ ]:
sol = SteadierCom(community, abundance=abundance, growth=growth, allocation=True, constraints=env, w_e=w_e, w_r=w_r, objective=target)

(-10.0, inf)